# 区县层面药店—医院周边商业—控制变量面板

以 `counties19` 的 2019 年区县边界为基准，构建 2012—2019 年完整区县面板。药店按 `PAC`、`county19` 和年份统计唯一 `shop_id`；医院数据按 `county_id` 和年份聚合医院数量、医院周边药店及餐馆数量的总和与均值；最后按 `county_id`、`year` 合并 `county_panel_1020.dta` 控制变量。

In [1]:
from pathlib import Path
import shutil
import struct

import numpy as np
import pandas as pd

ROOT = Path(r"D:\\Hospital_Pharmacy program")
PHARMACY_DIR = ROOT / "data" / "pharmacy" / "pharm_clean"
RESTAURANT_DIR = ROOT / "data" / "pharmacy" / "restaurant"
HOSPITAL_PATH = ROOT / "data" / "hospital_pharmacy" / "hosp_pharm_panel" / "hospital_pharm_panel.csv"
COUNTY_NEAREST_PUBLIC_PATH = ROOT / "data" / "hospital_pharmacy" / "hosp_pharm_panel" / "county_pharmacy_nearest_public_hospital_counts.csv"
COUNTY_DBF = ROOT / "data" / "county" / "controls" / "county_boundaries" / "counties19.dbf"
CONTROL_PATH = ROOT / "data" / "county" / "controls" / "county_panel_1020.dta"
OUTPUT_DIR = ROOT / "data" / "hospital_pharmacy" / "county_pharm_panel"
OUTPUT_CSV = OUTPUT_DIR / "county_pharm_panel.csv"
OUTPUT_DTA = OUTPUT_DIR / "county_pharm_panel.dta"
SUMMARY_CSV = OUTPUT_DIR / "county_pharm_panel_build_summary.csv"
SPATIAL_MATCH_CSV = OUTPUT_DIR / "county_pharm_panel_hospital_spatial_county_match.csv"
YEARS = list(range(2012, 2020))

def normalize_code(series):
    values = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    return values.where(values.str.fullmatch(r"\d+", na=False)).str.zfill(6)

def read_dbf(path):
    rows = []
    with path.open("rb") as handle:
        header = handle.read(32)
        record_count = struct.unpack("<I", header[4:8])[0]
        header_length = struct.unpack("<H", header[8:10])[0]
        record_length = struct.unpack("<H", header[10:12])[0]
        fields = []
        while True:
            first_byte = handle.read(1)
            if first_byte == b"\r":
                break
            descriptor = first_byte + handle.read(31)
            raw_name = descriptor[:11].split(b"\x00", 1)[0]
            fields.append((raw_name.decode("utf-8", "replace"), descriptor[16]))
        handle.seek(header_length)
        for _ in range(record_count):
            record = handle.read(record_length)
            if not record or record[:1] == b"\x1a":
                break
            if record[:1] == b"*":
                continue
            offset = 1
            row = {}
            for name, length in fields:
                row[name] = record[offset:offset + length].decode("utf-8", "replace").strip()
                offset += length
            rows.append(row)
    return pd.DataFrame(rows)

In [2]:
counties = read_dbf(COUNTY_DBF).rename(columns={
    "PAC": "county_id",
    "NAME": "county19",
    "省代码": "province_code",
    "省": "province19",
    "市代码": "city_code",
    "市": "city19",
    "类型": "county_type",
})
counties["county_id"] = normalize_code(counties["county_id"])
counties = counties[~counties["county_id"].str[:2].isin(["71", "81", "82"])].copy()
counties = counties[["county_id", "county19", "province_code", "province19", "city_code", "city19", "county_type"]]
counties["_name_present"] = counties["county19"].ne("").astype("int8")
counties = (counties.sort_values(["county_id", "_name_present"])
            .drop_duplicates("county_id", keep="last")
            .drop(columns="_name_present")
            .reset_index(drop=True))
assert len(counties) == 2853

panel = counties.merge(pd.DataFrame({"year": YEARS}), how="cross")
panel["year"] = panel["year"].astype("int16")
assert not panel.duplicated(["county_id", "year"]).any()
panel.shape

(22824, 8)

In [3]:
spatial_county_code_map = (
    counties.rename(columns={
        "province19": "spatial_province",
        "city19": "spatial_city",
        "county19": "spatial_county",
        "county_type": "spatial_county_type",
        "county_id": "spatial_county_id",
    })[["spatial_county_id", "spatial_province", "spatial_city", "spatial_county", "spatial_county_type"]]
    .copy()
)
for column in ["spatial_province", "spatial_city", "spatial_county", "spatial_county_type"]:
    spatial_county_code_map[column] = spatial_county_code_map[column].astype("string").str.strip()
spatial_county_code_map["spatial_county_id"] = normalize_code(spatial_county_code_map["spatial_county_id"])
assert not spatial_county_code_map.duplicated(["spatial_province", "spatial_city", "spatial_county", "spatial_county_type"]).any()
spatial_county_code_map.shape


(2853, 5)

In [4]:
pharmacy_parts = []
pharmacy_input_rows = 0
pharmacy_missing_pac = 0
for path in sorted(PHARMACY_DIR.glob("renew_pharmacy_dzdp_pharmacies_*.csv")):
    pharmacy = pd.read_csv(path, usecols=["year", "shop_id", "PAC", "county19"], dtype="string")
    pharmacy_input_rows += len(pharmacy)
    pharmacy["county_id"] = normalize_code(pharmacy["PAC"])
    pharmacy["county19"] = pharmacy["county19"].astype("string").str.strip()
    pharmacy["year"] = pd.to_numeric(pharmacy["year"], errors="coerce").astype("Int64")
    pharmacy_missing_pac += int(pharmacy["county_id"].isna().sum())
    pharmacy = pharmacy.dropna(subset=["county_id", "year", "shop_id"])
    counts = (pharmacy.groupby(["county_id", "county19", "year"], as_index=False)
              .agg(county_pharmacy_count=("shop_id", "nunique")))
    pharmacy_parts.append(counts)

pharmacy_panel = pd.concat(pharmacy_parts, ignore_index=True)
pharmacy_panel = (pharmacy_panel.groupby(["county_id", "year"], as_index=False)
                  .agg(county_pharmacy_count=("county_pharmacy_count", "sum")))
pharmacy_panel = pharmacy_panel[pharmacy_panel["county_id"].isin(counties["county_id"])].copy()
pharmacy_panel["pharmacy_observed"] = 1
assert pharmacy_panel["county_id"].isin(counties["county_id"]).all()
pharmacy_panel.groupby("year")["county_pharmacy_count"].sum()

year
2012     19433
2013    157952
2014    228700
2015    189435
2016    239612
2017    308308
2018    369032
2019    365325
Name: county_pharmacy_count, dtype: int64

In [5]:
restaurant_parts = []
restaurant_input_rows = 0
restaurant_missing_pac = 0
for path in sorted(RESTAURANT_DIR.glob("dzdp_restaurants_*.csv")):
    annual_parts = []
    for restaurant in pd.read_csv(
        path,
        usecols=["year", "shop_id", "PAC", "county19"],
        dtype="string",
        chunksize=250_000,
    ):
        restaurant_input_rows += len(restaurant)
        restaurant["county_id"] = normalize_code(restaurant["PAC"])
        restaurant["year"] = pd.to_numeric(restaurant["year"], errors="coerce").astype("Int64")
        restaurant_missing_pac += int(restaurant["county_id"].isna().sum())
        restaurant = restaurant.dropna(subset=["county_id", "year", "shop_id"])
        restaurant = restaurant[restaurant["year"].isin(YEARS)]
        restaurant = restaurant[restaurant["county_id"].isin(counties["county_id"])]
        shop_key = pd.util.hash_pandas_object(
            restaurant[["county_id", "year", "shop_id"]], index=False
        ).astype("uint64")
        annual_parts.append(pd.DataFrame({
            "county_id": restaurant["county_id"].to_numpy(),
            "year": restaurant["year"].to_numpy(),
            "shop_key": shop_key.to_numpy(),
        }))
    annual_restaurants = pd.concat(annual_parts, ignore_index=True).drop_duplicates("shop_key")
    annual_counts = (annual_restaurants.groupby(["county_id", "year"], as_index=False)
                     .agg(county_restaurant_count=("shop_key", "size")))
    restaurant_parts.append(annual_counts)

restaurant_panel = pd.concat(restaurant_parts, ignore_index=True)
restaurant_panel = (restaurant_panel.groupby(["county_id", "year"], as_index=False)
                    .agg(county_restaurant_count=("county_restaurant_count", "sum")))
restaurant_panel["restaurant_observed"] = 1
assert not restaurant_panel.duplicated(["county_id", "year"]).any()
restaurant_panel.groupby("year")["county_restaurant_count"].sum()

year
2012     116159
2013     409947
2014     564313
2015     764013
2016    1165879
2017    1281732
2018    1631997
2019    1572579
Name: county_restaurant_count, dtype: int64

In [6]:
# Keep the hospital-centered variables in hospital_pharm_panel.csv, but do not
# aggregate the hospital-level pharm_* counts into the county panel. The new
# pharmacy-centered counts are merged directly below with distinct names.
nearby_source_columns = [
    "chain_pharm_500m", "chain_pharm_1km", "chain_pharm_2km", "chain_pharm_3km", "chain_pharm_1_2km", "chain_pharm_2_3km",
    "resta_500m", "resta_1km", "resta_2km", "resta_3km", "resta_1_2km", "resta_2_3km",
]
hospital_columns = ["hospital_id", "year", "county_id", "policy_year", "public_flag", "private_flag"] + nearby_source_columns
hospitals = pd.read_csv(HOSPITAL_PATH, usecols=hospital_columns, dtype={"hospital_id": "string", "county_id": "string"})
hospital_input_rows = len(hospitals)
hospitals["county_id"] = normalize_code(hospitals["county_id"])
hospitals["year"] = pd.to_numeric(hospitals["year"], errors="coerce").astype("Int64")
numeric_columns = ["policy_year", "public_flag", "private_flag"] + nearby_source_columns
hospitals[numeric_columns] = hospitals[numeric_columns].apply(pd.to_numeric, errors="coerce")
hospitals = hospitals[hospitals["year"].isin(YEARS)].dropna(subset=["hospital_id", "county_id", "year"]).copy()
hospitals = hospitals[hospitals["county_id"].isin(counties["county_id"])].copy()
assert hospitals["county_id"].isin(counties["county_id"]).all()
assert not hospitals.duplicated(["hospital_id", "year"]).any()

policy_county = (hospitals.loc[(hospitals["public_flag"] == 1) & hospitals["policy_year"].notna(),
                               ["county_id", "policy_year"]]
                 .groupby("county_id", as_index=False)["policy_year"].min())
policy_county["policy_year"] = policy_county["policy_year"].round().astype("int16")
assert not policy_county.duplicated("county_id").any()

hospital_base = (hospitals.groupby(["county_id", "year"], as_index=False)
                 .agg(hospital_count=("hospital_id", "nunique"),
                      public_hospital_count=("public_flag", "sum"),
                      private_hospital_count=("private_flag", "sum")))
hospital_base["hospital_observed"] = 1

nearby_aggregations = {}
for column in nearby_source_columns:
    nearby_aggregations[f"nearby_sum_{column}"] = (column, "sum")
nearby_panel = hospitals.groupby(["county_id", "year"], as_index=False).agg(**nearby_aggregations)
hospital_panel = hospital_base.merge(nearby_panel, on=["county_id", "year"], how="left", validate="one_to_one")

nearest_public_count_columns = [
    "pharm_nearest_pub_total", "pharm_has_same_county_pub",
    "pharm_nearest_pub_le500m", "pharm_nearest_pub_gt500m",
    "pharm_nearest_pub_le1km", "pharm_nearest_pub_gt1km",
    "pharm_nearest_pub_le2km", "pharm_nearest_pub_gt2km",
]
nearest_public_panel = pd.read_csv(
    COUNTY_NEAREST_PUBLIC_PATH,
    usecols=["county_id", "year"] + nearest_public_count_columns,
    dtype={"county_id": "string"},
)
nearest_public_panel["county_id"] = normalize_code(nearest_public_panel["county_id"])
nearest_public_panel["year"] = pd.to_numeric(nearest_public_panel["year"], errors="coerce").astype("Int64")
nearest_public_panel[nearest_public_count_columns] = nearest_public_panel[nearest_public_count_columns].apply(
    pd.to_numeric, errors="coerce"
)
nearest_public_panel = nearest_public_panel[
    nearest_public_panel["year"].isin(YEARS)
    & nearest_public_panel["county_id"].isin(counties["county_id"])
].copy()
assert not nearest_public_panel.duplicated(["county_id", "year"]).any()
for scope in ["500m", "1km", "2km"]:
    assert (
        nearest_public_panel[f"pharm_nearest_pub_le{scope}"]
        + nearest_public_panel[f"pharm_nearest_pub_gt{scope}"]
        == nearest_public_panel["pharm_nearest_pub_total"]
    ).all()
assert (nearest_public_panel["pharm_nearest_pub_le500m"] <= nearest_public_panel["pharm_nearest_pub_le1km"]).all()
assert (nearest_public_panel["pharm_nearest_pub_le1km"] <= nearest_public_panel["pharm_nearest_pub_le2km"]).all()
hospital_panel.shape, nearest_public_panel.shape


(19401, 24)

In [7]:
controls = pd.read_stata(CONTROL_PATH, columns=["pac19", "year", "ln_nl_mean", "pop"], convert_categoricals=False)
controls = controls[controls["year"].between(min(YEARS), max(YEARS))].copy()
controls["county_id"] = normalize_code(controls["pac19"])
controls["year"] = pd.to_numeric(controls["year"], errors="coerce").astype("Int64")
controls["pop"] = pd.to_numeric(controls["pop"], errors="coerce")
controls["ln_pop"] = np.log(controls["pop"].where(controls["pop"].gt(0)))
controls = controls.drop(columns=["pac19"])
controls = controls[controls["county_id"].isin(counties["county_id"])]
assert not controls.duplicated(["county_id", "year"]).any()
controls["controls_matched"] = 1

final = panel.merge(pharmacy_panel, on=["county_id", "year"], how="left", validate="one_to_one")
final = final.merge(restaurant_panel, on=["county_id", "year"], how="left", validate="one_to_one")
final = final.merge(hospital_panel, on=["county_id", "year"], how="left", validate="one_to_one")
final = final.merge(nearest_public_panel, on=["county_id", "year"], how="left", validate="one_to_one")
final = final.merge(policy_county, on="county_id", how="left", validate="many_to_one")
final = final.merge(controls, on=["county_id", "year"], how="left", validate="one_to_one")

indicator_columns = ["pharmacy_observed", "restaurant_observed", "hospital_observed", "controls_matched"]
count_columns = ["county_pharmacy_count", "county_restaurant_count", "hospital_count", "public_hospital_count", "private_hospital_count"]
non_pharmacy_count_columns = ["county_restaurant_count", "hospital_count", "public_hospital_count", "private_hospital_count"]
nearby_columns = [column for column in final.columns if column.startswith("nearby_")]
final[indicator_columns] = final[indicator_columns].fillna(0).astype("int8")
final["county_pharmacy_count"] = pd.to_numeric(final["county_pharmacy_count"], errors="coerce")
final[non_pharmacy_count_columns] = final[non_pharmacy_count_columns].fillna(0).astype("int32")
final[nearby_columns] = final[nearby_columns].fillna(0.0)
assert final.loc[final["pharmacy_observed"].eq(0), "county_pharmacy_count"].isna().all()

nearby_rename = {column: column.removeprefix("nearby_sum_") for column in final.columns if column.startswith("nearby_sum_")}
final = final.rename(columns=nearby_rename)
remove_columns = [
    "controls_matched",
    "pop",
] + [column for column in final.columns if column.startswith("nearby_mean_")]
final = final.drop(columns=remove_columns, errors="ignore")
log_source_columns = [
    "county_pharmacy_count", "county_restaurant_count",
] + [column for column in nearby_source_columns if column.startswith(("pharm_", "resta_"))]
for column in log_source_columns:
    final[f"ln_{column}"] = np.log(final[column].where(final[column].gt(0)))

final = final.sort_values(["county_id", "year"]).reset_index(drop=True)
assert len(final) == len(counties) * len(YEARS)
assert not final.duplicated(["county_id", "year"]).any()
assert "controls_matched" not in final.columns
assert "pop" not in final.columns
assert not any(column.startswith("nearby_mean_") for column in final.columns)
assert not any(column.startswith("nearby_sum_") for column in final.columns)
assert "ln_pop" in final.columns
assert "county_restaurant_count" in final.columns
assert "restaurants_count" not in final.columns
assert "policy_year" in final.columns
assert all(column in final.columns for column in ["pharm_nearest_pub_le500m", "pharm_nearest_pub_gt2km", "chain_pharm_500m", "resta_500m"])
assert not any(column in final.columns for column in ["pharm_500m", "pharm_1km", "pharm_2km", "pharm_3km"])

expected_output_dir = ROOT / "data" / "hospital_pharmacy" / "county_pharm_panel"
assert OUTPUT_DIR.resolve() == expected_output_dir.resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def replace_existing_file(temp_path, target_path):
    if target_path.exists():
        target_path.unlink()
    temp_path.replace(target_path)

temp_csv = OUTPUT_CSV.with_suffix(".tmp.csv")
final.to_csv(temp_csv, index=False, encoding="utf-8-sig")
replace_existing_file(temp_csv, OUTPUT_CSV)

summary = pd.DataFrame({
    "metric": [
        "panel_rows", "counties", "years", "pharmacy_input_rows", "pharmacy_missing_pac",
        "pharmacy_count_total", "restaurant_input_rows", "restaurant_missing_pac",
        "restaurant_count_total", "hospital_input_rows", "hospital_rows",
        "counties_with_policy_year", "control_rows_matched", "ln_pop_missing", "pharmacy_observed_rows",
    ],
    "value": [
        len(final), final["county_id"].nunique(), final["year"].nunique(), pharmacy_input_rows, pharmacy_missing_pac,
        int(final["county_pharmacy_count"].sum()), restaurant_input_rows, restaurant_missing_pac,
        int(final["county_restaurant_count"].sum()), hospital_input_rows, len(hospitals),
        int(policy_county["county_id"].nunique()),
        int(controls[controls["county_id"].isin(counties["county_id"])].shape[0]), int(final["ln_pop"].isna().sum()),
        int(final["pharmacy_observed"].sum()),
    ],
})
summary.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
summary


,metric,value
0,panel_rows,22824
1,counties,2853
2,years,8
3,pharmacy_input_rows,1879909
4,pharmacy_missing_pac,2
5,pharmacy_count_total,1877797
6,restaurant_input_rows,7611569
7,restaurant_missing_pac,107
8,restaurant_count_total,7506619
9,hospital_input_rows,147783


In [8]:
# Build the two epidemiology-station IV variables from the original GB86 file and merge them into the panel.
IV_DIR = ROOT / "data" / "county" / "ivvar"
EPISXQ_PATH = IV_DIR / "episxq.dat"
IV_CROSSWALK_CSV = ROOT / "data" / "hospital_pharmacy" / "county_pharmacy_panel" / "crosswalk_gb86sxq_to_pac19_cnadmin.csv"
IV_GB86_OUTPUT_CSV = OUTPUT_DIR / "epi_50s_iv_gb86sxq.csv"
IV_PAC19_OUTPUT_CSV = OUTPUT_DIR / "epi_50s_iv_pac19_cnadmin.csv"
IV_MERGE_SUMMARY_CSV = OUTPUT_DIR / "county_pharm_panel_epi50s_iv_merge_summary.csv"

assert EPISXQ_PATH.exists(), f"Missing raw epidemiology-station file: {EPISXQ_PATH}"
assert IV_CROSSWALK_CSV.exists(), f"Missing GB86-to-2019 crosswalk: {IV_CROSSWALK_CSV}"

raw_epi = pd.read_csv(
    EPISXQ_PATH,
    sep=r"\s+",
    na_values=[-99999, "-99999"],
    dtype={"GB86SXQ": "string"},
)
raw_epi.columns = [column.upper() for column in raw_epi.columns]
required_epi_columns = ["GB86SXQ", "ECNT59", "EYRS59"]
missing_epi_columns = sorted(set(required_epi_columns) - set(raw_epi.columns))
assert not missing_epi_columns, f"Missing expected epidemiology-station columns: {missing_epi_columns}"

epi_gb86 = raw_epi[required_epi_columns].copy()
epi_gb86 = epi_gb86.rename(columns={
    "GB86SXQ": "gb86sxq",
    "ECNT59": "epi_cnt_1959",
    "EYRS59": "epi_yrs_50s",
})
epi_gb86["gb86sxq"] = normalize_code(epi_gb86["gb86sxq"])
for column in ["epi_cnt_1959", "epi_yrs_50s"]:
    epi_gb86[column] = pd.to_numeric(epi_gb86[column], errors="coerce")
epi_gb86["ln_epi_cnt_1959_1p"] = np.log1p(epi_gb86["epi_cnt_1959"].clip(lower=0))
epi_gb86["ln_epi_yrs_50s_1p"] = np.log1p(epi_gb86["epi_yrs_50s"].clip(lower=0))
assert not epi_gb86.duplicated("gb86sxq").any()
epi_gb86.to_csv(IV_GB86_OUTPUT_CSV, index=False, encoding="utf-8-sig")

crosswalk = pd.read_csv(IV_CROSSWALK_CSV, dtype="string")
crosswalk["gb86sxq"] = normalize_code(crosswalk["gb86sxq"])
county_id_source = "county_id_2019" if "county_id_2019" in crosswalk.columns else "county_id"
crosswalk["county_id"] = normalize_code(crosswalk[county_id_source])
crosswalk["cw_weight"] = pd.to_numeric(crosswalk["cw_weight"] if "cw_weight" in crosswalk.columns else 1, errors="coerce").fillna(1.0)
crosswalk = crosswalk[["gb86sxq", "county_id", "cw_weight"]].dropna(subset=["gb86sxq", "county_id"])

epi_crosswalked = crosswalk.merge(epi_gb86, on="gb86sxq", how="left", validate="many_to_one")
missing_raw_epi = int(epi_crosswalked["epi_cnt_1959"].isna().sum())
for column in ["epi_cnt_1959", "epi_yrs_50s"]:
    epi_crosswalked[f"weighted_{column}"] = epi_crosswalked[column] * epi_crosswalked["cw_weight"]

epi_pac19 = epi_crosswalked.groupby("county_id", as_index=False).agg(
    epi_cnt_1959=("weighted_epi_cnt_1959", "sum"),
    epi_yrs_50s=("weighted_epi_yrs_50s", "sum"),
)
epi_pac19["ln_epi_cnt_1959_1p"] = np.log1p(epi_pac19["epi_cnt_1959"].clip(lower=0))
epi_pac19["ln_epi_yrs_50s_1p"] = np.log1p(epi_pac19["epi_yrs_50s"].clip(lower=0))
assert not epi_pac19.duplicated("county_id").any()
epi_pac19.to_csv(IV_PAC19_OUTPUT_CSV, index=False, encoding="utf-8-sig")

iv_columns = ["epi_cnt_1959", "epi_yrs_50s", "ln_epi_cnt_1959_1p", "ln_epi_yrs_50s_1p"]
panel_rows_before_iv = len(final)
panel_keys_before_iv = final[["county_id", "year"]].copy()
final = final.drop(columns=[column for column in iv_columns if column in final.columns], errors="ignore")
final = final.merge(epi_pac19, on="county_id", how="left", validate="many_to_one")

assert len(final) == panel_rows_before_iv
assert final[["county_id", "year"]].equals(panel_keys_before_iv)
assert not final.duplicated(["county_id", "year"]).any()

iv_summary = pd.DataFrame({
    "metric": [
        "raw_epi_gb86_rows", "crosswalk_rows", "crosswalk_rows_missing_raw_epi",
        "pac19_iv_counties", "panel_counties_with_iv", "panel_counties_without_iv",
        "panel_rows_with_iv", "panel_rows_without_iv",
    ],
    "value": [
        len(epi_gb86), len(crosswalk), missing_raw_epi,
        epi_pac19["county_id"].nunique(),
        final.loc[final["epi_cnt_1959"].notna(), "county_id"].nunique(),
        final.loc[final["epi_cnt_1959"].isna(), "county_id"].nunique(),
        int(final["epi_cnt_1959"].notna().sum()),
        int(final["epi_cnt_1959"].isna().sum()),
    ],
})
iv_summary.to_csv(IV_MERGE_SUMMARY_CSV, index=False, encoding="utf-8-sig")
iv_summary


,metric,value
0,raw_epi_gb86_rows,2398
1,crosswalk_rows,2698
2,crosswalk_rows_missing_raw_epi,54
3,pac19_iv_counties,2556
4,panel_counties_with_iv,2359
5,panel_counties_without_iv,494
6,panel_rows_with_iv,18872
7,panel_rows_without_iv,3952


In [9]:
VARIABLE_LABELS = {
    "county_id": "2019 county administrative code",
    "county19": "2019 county name",
    "province_code": "Province administrative code",
    "province19": "2019 province name",
    "city_code": "Prefecture administrative code",
    "city19": "2019 prefecture name",
    "county_type": "County administrative type",
    "year": "Calendar year",
    "county_pharmacy_count": "Total pharmacies in county",
    "pharmacy_observed": "County pharmacy source observed",
    "county_restaurant_count": "Total restaurants in county",
    "restaurant_observed": "County restaurant source observed",
    "hospital_count": "Total hospitals in county",
    "public_hospital_count": "Public hospitals in county",
    "private_hospital_count": "Private hospitals in county",
    "hospital_observed": "County hospital source observed",
    "policy_year": "Earliest public-hospital reform year in county",
    "ln_nl_mean": "Natural log of mean nighttime lights plus one",
    "ln_pop": "Natural log of county population",
    "pharm_nearest_pub_total": "Geocoded pharmacies used for nearest-public-hospital counts",
    "pharm_has_same_county_pub": "Pharmacies with an active geocoded public hospital in same county",
    "pharm_nearest_pub_le500m": "Unique pharmacies <=500 m from nearest same-county public hospital",
    "pharm_nearest_pub_gt500m": "Unique pharmacies >500 m from all same-county public hospitals",
    "pharm_nearest_pub_le1km": "Unique pharmacies <=1 km from nearest same-county public hospital",
    "pharm_nearest_pub_gt1km": "Unique pharmacies >1 km from all same-county public hospitals",
    "pharm_nearest_pub_le2km": "Unique pharmacies <=2 km from nearest same-county public hospital",
    "pharm_nearest_pub_gt2km": "Unique pharmacies >2 km from all same-county public hospitals",
}
scope_labels = {
    "500m": "within 500 m of hospitals",
    "1km": "within 1 km of hospitals",
    "2km": "within 2 km of hospitals",
    "3km": "within 3 km of hospitals",
    "1_2km": "in 1-2 km hospital rings",
    "2_3km": "in 2-3 km hospital rings",
}
VARIABLE_LABELS.update({
    "epi_cnt_1959": "Epidemiology-station stock in 1959, GB86 crosswalked",
    "epi_yrs_50s": "Epidemiology station-years accumulated by 1959, GB86 crosswalked",
    "ln_epi_cnt_1959_1p": "Natural log of 1959 epidemiology-station stock plus one",
    "ln_epi_yrs_50s_1p": "Natural log of 1950s epidemiology station-years plus one",
})
for column in nearby_source_columns:
    if column.startswith("chain_pharm_"):
        item = "Chain pharmacies"
        scope = column.removeprefix("chain_pharm_")
    elif column.startswith("pharm_"):
        item = "Pharmacies"
        scope = column.removeprefix("pharm_")
    else:
        item = "Restaurants"
        scope = column.removeprefix("resta_")
    VARIABLE_LABELS[column] = f"Sum of {item.lower()} {scope_labels[scope]}"
for column in log_source_columns:
    VARIABLE_LABELS[f"ln_{column}"] = f"Natural log of {VARIABLE_LABELS[column].lower()}"

missing_labels = sorted(set(final.columns) - set(VARIABLE_LABELS))
assert not missing_labels, f"Variables without labels: {missing_labels}"
assert all(len(label) <= 80 for label in VARIABLE_LABELS.values())

stata_output = final.copy()
for column in stata_output.select_dtypes(include=["object", "string"]).columns:
    stata_output[column] = stata_output[column].fillna("").astype(str)
value_labels = {column: {0: "No", 1: "Yes"} for column in indicator_columns if column != "controls_matched"}
temp_dta = OUTPUT_DTA.with_suffix(".tmp.dta")
stata_output.to_stata(
    temp_dta,
    write_index=False,
    version=118,
    data_label="County pharmacy restaurant hospital panel, 2012-2019",
    variable_labels=VARIABLE_LABELS,
    value_labels=value_labels,
)
replace_existing_file(temp_dta, OUTPUT_DTA)
pd.DataFrame({"variable": final.columns, "label": [VARIABLE_LABELS[column] for column in final.columns]})

,variable,label
0,county_id,2019 county administrative code
1,county19,2019 county name
2,province_code,Province administrative code
3,province19,2019 province name
4,city_code,Prefecture administrative code
5,city19,2019 prefecture name
6,county_type,County administrative type
7,year,Calendar year
8,county_pharmacy_count,Total pharmacies in county
9,pharmacy_observed,County pharmacy source observed
